**Loan Default Prediction System**

In [ ]:
#Import necesassry libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
sns.set_style('whitegrid')

In [ ]:
# Load the data
df = pd.read_csv('/content/Loan_default.csv')


In [ ]:
df = df.drop(columns=['LoanID'])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.describe()

In [ ]:
Target balance — what % actually default?
rate = df['Default'].mean()
print(f'Default rate: {rate:.1%}')
df['Default'].value_counts().plot(kind='bar', color=['#2c7fb8','#e34a33'])
plt.title('0 = repaid,  1 = defaulted'); plt.show()
# NOTE: only ~12% default. The data is imbalanced — keep an eye on F1, not just accuracy.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.kdeplot(data=df, x='CreditScore', hue='Default', common_norm=False, ax=ax[0])
ax[0].set_title('Credit Score')
sns.kdeplot(data=df, x='InterestRate', hue='Default', common_norm=False, ax=ax[1])
ax[1].set_title('Interest Rate')
plt.show()

In [ ]:
# correction
num = ['Age','Income','LoanAmount','CreditScore','MonthsEmployed',
       'NumCreditLines','InterestRate','LoanTerm','DTIRatio']
plt.figure(figsize=(9,7))
sns.heatmap(df[num].corr(), annot=True, cmap='Blues')
plt.show()


In [ ]:
# correction
num = ['Age','Income','LoanAmount','CreditScore','MonthsEmployed',
       'NumCreditLines','InterestRate','LoanTerm','DTIRatio']
plt.figure(figsize=(9,7))
sns.heatmap(df[num+['Default']].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Numeric correlation'); plt.show()

In [ ]:
df.head()

One-hot encoding

In [ ]:
df_enc = pd.get_dummies(df, drop_first=True)
# Drop rows where 'Default' is NaN before splitting
df_enc.dropna(subset=['Default'], inplace=True)
X = df_enc.drop(columns=['Default'])
y = df_enc['Default']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('Train:', X_train.shape, ' Test:', X_test.shape)

In [29]:
# Train and evaluate three models
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(), random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results[name] = {'accuracy': acc, 'f1': f1}
    print(f'{name}: Accuracy={acc:.3f}, F1={f1:.3f}')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: Accuracy=0.679, F1=0.335
Random Forest: Accuracy=0.885, F1=0.026
XGBoost: Accuracy=0.714, F1=0.337


## Key Concept: Encoding & Decoding Categorical Variables

Machine learning models require numeric input. This dataset contains several
categorical text columns (Education, EmploymentType, MaritalStatus, LoanPurpose,
HasMortgage, HasDependents, HasCoSigner) which must be converted to numbers before
training.

We use **One-Hot Encoding** (`pd.get_dummies`, `drop_first=True`) to transform
these columns into binary (0/1) numeric features — this is the **encoding** step.

After the model makes predictions, we interpret the output (0 or 1) back as
"No Default" or "Default" — this is the **decoding** step.